# FluxQuery · Step 1 · Pre-processing of omics data

FluxQuery runs in four steps:

| Step | Language | What it does | Code |
|---|---|---|---|
| **1** | **Python** | **turns public omics data into XomicsToModel inputs** | **`preprocessingFluxQuery.ipynb` (this notebook)** |
| 2 | MATLAB | builds the cholinergic and glutamatergic models with XomicsToModel | `modelGenFluxQuery.m` |
| 3 | MATLAB | samples the flux space of the three neuron models | `samplesGenFluxQuery.m` |
| 4 | Python | learns the sampled flux spaces, queries them for axonal growth and designs a medium | `samplingAnalysisFluxQuery.ipynb` |

This notebook produces, in the project's `data/omics` folder:

* `transcriptomicData_<neuron>Neuron.txt`: `genes,expVal` (Entrez ID, FPKM), the format of `data/formatExamples/transcriptomicData.txt`;
* `exometabolomicData_glutamatergicNeuron.txt`: `rxns,mets,metName,platform,mean,SD` (umol/gDW/h), the format of `data/formatExamples/exometabolomicData.txt`;
* `exometabolomicDirection_glutamatergicNeuron.txt`: metabolites that are clearly released or consumed but cannot be given an absolute rate.

Downloaded raw files are cached in `data/omics/<neuron>/raw`, so the notebook downloads each dataset only once. If a cached file is present, it is used instead of the internet.

**Sources**

| Neuron | Transcriptomics | Exometabolomics |
|---|---|---|
| cholinergic (basal forebrain) | GEO GSE308143, untreated controls NT1_C-NT5_C, day 60-70 (Russo lab, *Biomolecules* 2025) | none published |
| glutamatergic (cortical) | GEO GSE112732, NGN2 patterned induced neurons, day 49, 2 lines x 3 (Nehme *et al.*, *Cell Rep* 2018) | Metabolomics Workbench ST004403, isogenic control, week 4 (Spinelli lab, *Cell Rep* 2026) |
| dopaminergic | not needed: the published iDopaNeuro model is used; GSE308143 NT_D is kept as a reference | |

## 0. Settings

Leave `PROJECT_DIR` as `None` to choose the project folder (the one that contains `data/` and `results/`) in a dialog.

In [ ]:
PROJECT_DIR = None          # folder that contains data/ and results/
SUMMARY = "mean"            # one value per gene: "mean" of the control replicates (recommended), "median", or a sample name
ANALYTICAL_RSD = 0.10       # relative SD of a single LC-MS measurement, used in the flux uncertainty

In [ ]:
import gzip, io, os, re, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def select_folder(title, env_var=None):
    if env_var and os.environ.get(env_var):
        return Path(os.environ[env_var]).expanduser()
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
        sel = filedialog.askdirectory(title=title); root.destroy()
        if sel:
            return Path(sel)
    except Exception:
        pass
    return Path(input(f"{title}: ").strip().strip('"')).expanduser()

PROJECT_DIR = Path(PROJECT_DIR or select_folder("Select the FluxQuery project folder (contains data/ and results/)", "FQ_PROJECT"))
DATA = PROJECT_DIR / "data"
OMICS = DATA / "omics"
for n in ("cholinergic", "glutamatergic", "dopaminergic"):
    (OMICS / n / "raw").mkdir(parents=True, exist_ok=True)
print("project:", PROJECT_DIR)

### Helper functions
Downloads (GEO, MyGene.info, Metabolomics Workbench), FPKM calculation and file writers. The cell is collapsed; open it to read the code.

In [ ]:
GEO_FILE = "https://www.ncbi.nlm.nih.gov/geo/download/?acc={acc}&format=file&file={file}"
GEO_SOFT = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={acc}&targ=self&form=text&view=brief"
MYGENE = "https://mygene.info/v3"
MWTAB = "https://www.metabolomicsworkbench.org/rest/study/study_id/{study}/mwtab/txt"


def http_get(url, **kw):
    import requests
    r = requests.get(url, timeout=300, **kw)
    r.raise_for_status()
    return r


def as_text(data: bytes) -> str:
    return (gzip.decompress(data) if data[:2] == b"\x1f\x8b" else data).decode()


def cached(path: Path, fetch):
    # Return the content of `path`; download it with `fetch()` (bytes) the first time.
    if not path.exists():
        path.write_bytes(fetch())
        print(f"  downloaded {path.name} ({path.stat().st_size / 1e6:.1f} MB)")
    return path.read_bytes()


# ---------------------------------------------------------------- transcriptomics
def geo_sample_quant(gsm: str, raw: Path) -> tuple:
    # Salmon quant.sf attached to a GEO sample -> (sample title, text)
    soft = cached(raw / f"{gsm}.soft.txt", lambda: http_get(GEO_SOFT.format(acc=gsm)).content).decode()
    title = re.search(r"!Sample_title = (.*)", soft).group(1).strip()
    fname = re.search(r"!Sample_supplementary_file_1 = (\S+)", soft).group(1).rsplit("/", 1)[-1]
    data = cached(raw / fname, lambda: http_get(GEO_FILE.format(acc=gsm, file=fname)).content)
    return title, as_text(data)


def tx2gene_mygene(raw: Path) -> pd.Series:
    # Ensembl transcript -> Entrez gene for all human genes (MyGene.info), cached as tsv
    path = raw / "tx2entrez_human_mygene.tsv"
    if not path.exists():
        import requests
        rows, params = [], {"q": "_exists_:entrezgene", "species": "human", "fetch_all": "true",
                            "fields": "entrezgene,ensembl.transcript", "size": 1000}
        r = requests.get(f"{MYGENE}/query", params=params, timeout=300).json()
        while r.get("hits"):
            for d in r["hits"]:
                ens = d.get("ensembl") or []
                for e in ens if isinstance(ens, list) else [ens]:
                    tx = e.get("transcript") or []
                    for t in tx if isinstance(tx, list) else [tx]:
                        rows.append((t, str(d["entrezgene"])))
            r = requests.get(f"{MYGENE}/query", params={"scroll_id": r["_scroll_id"]}, timeout=300).json()
        pd.DataFrame(rows, columns=["transcript", "entrez"]).drop_duplicates("transcript").to_csv(path, sep="\t", index=False)
    t = pd.read_csv(path, sep="\t", dtype=str)
    return pd.Series(t.entrez.values, index=t.transcript.values)


def symbols2entrez_mygene(ids, raw: Path) -> pd.DataFrame:
    # gene symbols -> Entrez (official symbol preferred over aliases), cached as csv
    path = raw / "symbol2entrez_mygene.csv"
    if not path.exists():
        import requests
        out = {}
        ids = list(dict.fromkeys(map(str, ids)))
        for i in range(0, len(ids), 1000):
            res = requests.post(f"{MYGENE}/query", timeout=300, data={
                "q": ",".join(ids[i:i + 1000]), "scopes": "symbol,alias,ensembl.gene,entrezgene",
                "fields": "entrezgene,symbol", "species": "human"}).json()
            for d in res:
                if d.get("notfound") or "entrezgene" not in d:
                    continue
                exact = d.get("symbol") == d["query"]
                if d["query"] not in out or (exact and not out[d["query"]][1]):
                    out[d["query"]] = (str(d["entrezgene"]), exact)
        pd.DataFrame([(k, *v) for k, v in out.items()], columns=["id", "entrez", "exact"]).to_csv(path, index=False)
    m = pd.read_csv(path, dtype=str)
    m["exact"] = m["exact"].isin(["1", "True", "true"])
    return m.set_index("id")


def salmon_gene_fpkm(text: str, tx2gene: pd.Series) -> pd.Series:
    # gene FPKM = sum over the gene's transcripts of NumReads / (EffectiveLength x total reads) x 1e9
    q = pd.read_csv(io.StringIO(text), sep="\t")
    q["tx"] = q["Name"].str.split("|").str[0].str.split(".").str[0]
    total = q["NumReads"].sum()
    q = q[q["EffectiveLength"] > 0]
    q["fpkm"] = q["NumReads"] / q["EffectiveLength"] / total * 1e9
    q["gene"] = q["tx"].map(tx2gene)
    return q.dropna(subset=["gene"]).groupby("gene")["fpkm"].sum()


def table_to_entrez(T: pd.DataFrame, idmap: pd.DataFrame) -> pd.DataFrame:
    # gene-level table (rows = symbols) -> rows = Entrez; alias-only rows are dropped when the gene
    # also appears under its official symbol, the rest are summed per gene
    M = T.copy()
    M["entrez"] = M.index.map(idmap["entrez"])
    M["exact"] = M.index.map(idmap["exact"]).fillna(False).astype(bool)
    has_exact = M.loc[M["exact"], "entrez"].dropna().unique()
    M = M[M["entrez"].notna() & (M["exact"] | ~M["entrez"].isin(has_exact))]
    return M.drop(columns="exact").groupby("entrez").sum()


def summarise(M: pd.DataFrame, how: str = "mean") -> pd.Series:
    return M.mean(axis=1) if how == "mean" else M.median(axis=1) if how == "median" else M[how]


def write_transcriptomic(expr: pd.Series, out: Path):
    expr = expr.dropna()
    expr.index = expr.index.astype(int)
    expr = expr[~expr.index.duplicated()].sort_index()
    with open(out, "w") as fh:
        fh.write("genes,expVal\n")
        for g, v in expr.items():
            fh.write(f"{g},{repr(float(v))}\n")
    print(f"  wrote {out.relative_to(PROJECT_DIR)}: {len(expr):,} genes, {int((expr >= 1).sum()):,} with FPKM >= 1")


# ---------------------------------------------------------------- exometabolomics
def read_mwtab(text: str) -> pd.DataFrame:
    blocks = []
    for m in re.finditer(r"MS_METABOLITE_DATA_START\n(.*?)\nMS_METABOLITE_DATA_END", text, re.S):
        lines = [l for l in m.group(1).split("\n") if not l.startswith("Factors")]
        blocks.append(pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", index_col=0))
    D = pd.concat(blocks)
    return D[~D.index.duplicated()]


def culture_parameters(bibliomic: Path) -> dict:
    c = pd.read_excel(bibliomic, sheet_name="cellCultureData").iloc[0]
    p = dict(volume_L=float(c["volume"]), interval_h=float(c["interval"]),
             protein_g_per_L=float(c["averageProteinConcentration"]), assay_volume_L=float(c["assayVolume"]),
             protein_fraction=float(c["proteinFraction"]))
    p["gDW"] = p["protein_g_per_L"] * p["assay_volume_L"] / p["protein_fraction"]
    p["k"] = p["volume_L"] / (p["interval_h"] * p["gDW"])          # umol/L -> umol/gDW/h
    return p

## 1. Transcriptomics

**One expression value per gene.** XomicsToModel needs a single value per gene and condition. All samples used here are biological replicates of the same untreated condition, so their **mean** is the best estimate of the condition and is less noisy than any single sample. Choosing one replicate would discard information without removing bias, and the median adds noise with 4-6 replicates. `SUMMARY` changes this choice. Per-sample values are kept in `raw/` for inspection.

### 1.1 Cholinergic neurons: GSE308143 (Salmon quantification)

In [ ]:
def salmon_dataset(neuron, samples, tag):
    raw = OMICS / neuron / "raw"
    per_sample = raw / f"perSampleFPKM_{tag}.csv"
    if per_sample.exists():                                   # computed before: reuse
        M = pd.read_csv(per_sample, index_col=0)
    else:
        tx2gene = tx2gene_mygene(OMICS)
        cols = {}
        for gsm in samples:
            title, txt = geo_sample_quant(gsm, raw)
            cols[f"{gsm}_{title}"] = salmon_gene_fpkm(txt, tx2gene)
        M = pd.DataFrame(cols).fillna(0.0)
        M.index.name = "entrez"
        M.to_csv(per_sample)
    print(f"{neuron}: {M.shape[1]} replicates, {len(M):,} genes")
    return M

chol = salmon_dataset("cholinergic", ["GSM9238701", "GSM9238702", "GSM9238703", "GSM9238704", "GSM9238705"],
                      "GSE308143_cholinergic")
write_transcriptomic(summarise(chol, SUMMARY), OMICS / "cholinergic" / "transcriptomicData_cholinergicNeuron.txt")

dopa = salmon_dataset("dopaminergic", ["GSM9238682", "GSM9238683", "GSM9238684", "GSM9238685"],
                      "GSE308143_dopaminergic")
write_transcriptomic(summarise(dopa, SUMMARY),
                     OMICS / "dopaminergic" / "transcriptomicData_dopaminergicNeuron_referenceGSE308143.txt")

### 1.2 Glutamatergic neurons: GSE112732 (authors' FPKM table, day 49)

In [ ]:
raw = OMICS / "glutamatergic" / "raw"
per_sample = raw / "perSampleFPKM_GSE112732_D49.csv"
if per_sample.exists():
    glut = pd.read_csv(per_sample, index_col=0)
else:
    fname = "GSE112732_iPS1_iPS2_FPKMs.txt.gz"
    txt = as_text(cached(raw / fname, lambda: http_get(GEO_FILE.format(acc="GSE112732", file=fname)).content))
    T = pd.read_csv(io.StringIO(txt), sep="\t", index_col=0)
    T.index = T.index.astype(str).str.strip('"'); T.columns = [c.strip('"') for c in T.columns]
    T = T[[c for c in T.columns if re.search(r"\.D49\.", c)]]
    glut = table_to_entrez(T, symbols2entrez_mygene(T.index, raw))
    glut.index.name = "entrez"
    glut.to_csv(per_sample)
print(f"glutamatergic: {glut.shape[1]} replicates, {len(glut):,} genes")
write_transcriptomic(summarise(glut, SUMMARY), OMICS / "glutamatergic" / "transcriptomicData_glutamatergicNeuron.txt")

### 1.3 Quality control

Each culture is checked against identity and maturity markers. Basal forebrain cholinergic identity is LHX8, NKX2-1 and ISL1; cortical glutamatergic identity is SLC17A6 and SLC17A7. Mature neurons express MAP2 and SYP. Progenitors express SOX2, NES and MKI67, and glia express GFAP and S100B. Replicates should agree with each other.

In [ ]:
markers = {"CHAT": 1103, "SLC18A3": 6572, "LHX8": 431707, "NKX2-1": 7080, "ISL1": 3670, "SLC17A6": 57084,
           "SLC17A7": 57030, "GAD1": 2571, "TH": 7054, "MAP2": 4133, "SYP": 6855, "SOX2": 6657, "NES": 10763,
           "MKI67": 4288, "GFAP": 2670, "S100B": 6285}
qc = pd.DataFrame({n: [summarise(M, SUMMARY).reindex([g]).fillna(np.nan).iloc[0] for g in markers.values()]
                   for n, M in [("cholinergic", chol), ("glutamatergic", glut), ("dopaminergic (reference)", dopa)]},
                  index=list(markers))
cv = {n: float((M.std(axis=1) / M.mean(axis=1)).replace([np.inf], np.nan).median())
      for n, M in [("cholinergic", chol), ("glutamatergic", glut), ("dopaminergic (reference)", dopa)]}
print("median coefficient of variation between replicates:", {k: f"{v:.0%}" for k, v in cv.items()})
qc.round(2)

## 2. Exometabolomics: glutamatergic neurons (ST004403)

The conversion follows the dopaminergic study (Preciat *et al.*, bioRxiv 2021.06.30.450562). The exchange rate is assumed constant between the medium change and sampling:

$$v_i = \frac{(C_{\mathrm{spent},i} - C_{\mathrm{fresh},i})\,V}{\Delta t \cdot \mathrm{gDW}}, \qquad \mathrm{gDW} = \frac{\text{protein per well}}{\text{protein fraction of dry weight}}$$

The rate is in µmol/gDW/h; negative values are uptake. ST004403 reports LC-MS peak areas of spent and fresh medium measured in the same batch, so the fractional change $f_i = A_{\mathrm{spent}}/A_{\mathrm{fresh}} - 1$ is quantitative. With $C_{\mathrm{fresh},i}$ taken from the medium composition (`mediaData` of the bibliomic workbook):

$$C_{\mathrm{spent},i} - C_{\mathrm{fresh},i} = f_i\,C_{\mathrm{fresh},i}$$

V, Δt and gDW are the culture parameters of the bibliomic workbook (`cellCultureData`). The standard deviation combines the spread of the three replicates with `ANALYTICAL_RSD` on each measurement. Classification follows the article: *secreted* if mean − SD > 0, *uptaken* if mean + SD < 0, otherwise *unchanged*.

Two kinds of metabolite cannot get an absolute rate and are written to the direction file:

* metabolites absent from the medium;
* metabolites with a **net release**, because the amount released does not scale with the fresh-medium concentration.

In [ ]:
# curated metabolite -> Recon3D exchange map for ST004403: (metabolite, exchange, use, note)
METABOLITE_MAP = pd.DataFrame([
    ('1-Methyl nicotinamide', 'EX_1mncam[e]', True, ''),
    ('5-Amino-levulinic acid', 'EX_5aop[e]', True, ''),
    ('Acetylcholine', 'EX_ach[e]', False, 'trace signal only in spent medium; not used as a constraint'),
    ('Alanine', 'EX_ala_L[e]', False, 'not resolved from beta-alanine/D-alanine/sarcosine; also released by hydrolysis of the alanyl-glutamine dipeptide (GlutaMAX) in the medium -> not used as a constraint'),
    ('alpha-Aminoadipic acid', 'EX_L2aadp[e]', True, ''),
    ('Arginine', 'EX_arg_L[e]', True, ''),
    ('Asparagine', 'EX_asn_L[e]', True, ''),
    ('Betaine', 'EX_glyb[e]', True, ''),
    ('Biotin', 'EX_btn[e]', True, ''),
    ('Carnitine', 'EX_crn[e]', True, ''),
    ('Choline', 'EX_chol[e]', True, ''),
    ('Choline phosphate', 'EX_cholp[e]', False, 'phosphocholine is intracellular; appearance in the medium most likely reflects cell leakage/lysis -> not used'),
    ('Citrulline', 'EX_citr_L[e]', True, ''),
    ('Creatine', 'EX_creat[e]', True, ''),
    ('Creatinine', 'EX_crtn[e]', True, ''),
    ('Cystathionine', 'EX_cyst_L[e]', True, ''),
    ('Cystine', 'EX_Lcystin[e]', True, ''),
    ('Cytosine', 'EX_csn[e]', True, ''),
    ('Glucosamine', 'EX_gam[e]', True, ''),
    ('Glutamine', 'EX_gln_L[e]', False, 'fresh medium contains almost no free glutamine (dipeptide supplement); the increase reflects dipeptide hydrolysis -> not used as a constraint'),
    ('Glycine', 'EX_gly[e]', True, ''),
    ('Guanidoacetic acid', 'EX_gudac[e]', True, ''),
    ('Guanine', 'EX_gua[e]', True, ''),
    ('Hypoxanthine', 'EX_hxan[e]', True, 'purine degradation product; plausible secretion'),
    ('Isoleucine', 'EX_ile_L[e]', True, ''),
    ('Kynurenine', 'EX_Lkynr[e]', True, ''),
    ('Leucine', 'EX_leu_L[e]', True, ''),
    ('Lysine', 'EX_lys_L[e]', True, ''),
    ('Methionine', 'EX_met_L[e]', True, ''),
    ('N-6-Trimethyllysine', 'EX_tmlys[e]', True, ''),
    ('N-Acetyl-D-glucosamine', 'EX_acgam[e]', True, ''),
    ('N-Acetylneuraminic acid', 'EX_acnam[e]', True, ''),
    ('Niacinamide', 'EX_ncam[e]', True, ''),
    ('Ornithine', 'EX_orn[e]', True, ''),
    ('Oxidized glutathione', 'EX_gthox[e]', True, ''),
    ('Pantothenic acid', 'EX_pnto_R[e]', True, ''),
    ('Phenylalanine', 'EX_phe_L[e]', False, 'not detected in fresh medium although the medium contains phenylalanine (detection artefact) -> not used'),
    ('Proline', 'EX_pro_L[e]', True, ''),
    ('Pyridoxal', 'EX_pydx[e]', True, 'medium vitamin B6 (pyridoxine) is converted by cells and released as pyridoxal'),
    ('Riboflavin', 'EX_ribflv[e]', True, ''),
    ('Serine', 'EX_ser_L[e]', True, ''),
    ('Taurine', 'EX_taur[e]', True, ''),
    ('Thiamine', 'EX_thm[e]', True, ''),
    ('Threonine', 'EX_thr_L[e]', True, 'not resolved from homoserine/allothreonine (identical values)'),
    ('Tryptophan', 'EX_trp_L[e]', True, ''),
    ('Tyrosine', 'EX_tyr_L[e]', True, 'apparent +11% within measurement noise of a single blank'),
    ('Valine', 'EX_val_L[e]', True, ''),
    ('2-Hydroxybutyric acid', 'EX_2hb[e]', True, ''),
    ('2-Oxoglutaric acid', 'EX_akg[e]', True, ''),
    ('3-Methyl-2-oxovaleric acid', 'EX_3mop[e]', True, ''),
    ('Aspartic acid', 'EX_asp_L[e]', True, 'apparent +24% with a low blank signal; treat as unchanged'),
    ('Citric acid', 'EX_cit[e]', True, 'not resolved from isocitrate'),
    ('Folic acid', 'EX_fol[e]', False, 'apparent increase (+63%) is implausible; treat as unchanged'),
    ('Fumaric acid', 'EX_fum[e]', True, ''),
    ('gamma-Aminobutyric acid', 'EX_4abut[e]', True, ''),
    ('Glucose', 'EX_glc_D[e]', True, 'hexose signal; glucose, galactose, psicose and sorbose are not resolved (identical values); HILIC-positive sensitivity for hexoses is low, so the -13% is a lower bound on consumption'),
    ('Glutamic acid', 'EX_glu_L[e]', True, 'not resolved from N-methyl-D-aspartate (identical values)'),
    ('Guanosine', 'EX_gsn[e]', True, ''),
    ('Homovanillic acid', 'EX_homoval[e]', True, ''),
    ('Hydroxyphenyllactic acid', 'EX_34hpl[e]', True, ''),
    ('Lactic acid', 'EX_lac_L[e]', True, ''),
    ('Malic acid', 'EX_mal_L[e]', True, ''),
    ('N-Acetylaspartic acid', 'EX_Nacasp[e]', True, 'neuron-specific metabolite (NAT8L); secretion is a hallmark of neurons'),
    ('N-Acetylglutamic acid', 'EX_acglu[e]', True, ''),
    ('Oxalic acid', 'EX_oxa[e]', True, ''),
    ('Pyroglutamic acid', 'EX_5oxpro[e]', True, ''),
    ('Pyruvic acid', 'EX_pyr[e]', True, ''),
    ('Succinic acid', 'EX_succ[e]', True, ''),
    ('Uridine', 'EX_uri[e]', True, ''),
    ('Xanthosine', 'EX_xtsn[e]', True, '')
], columns=["metabolite", "rxns", "use", "note"]).set_index("metabolite")
print(len(METABOLITE_MAP), "metabolites mapped;", int(METABOLITE_MAP.use.sum()), "used")

In [ ]:
raw = OMICS / "glutamatergic" / "raw"
D = read_mwtab(cached(raw / "ST004403.mwtab.txt", lambda: http_get(MWTAB.format(study="ST004403")).content).decode())
fresh = pd.to_numeric(D["media blank 1"], errors="coerce")
spent_cols = [c for c in D.columns if c.startswith("media iControl")]
S = D[spent_cols].apply(pd.to_numeric, errors="coerce").add(fresh, axis=0)   # stored as spent - fresh
R = S.div(fresh.where(fresh > 0), axis=0)                                     # spent / fresh per replicate
f = R.mean(axis=1) - 1
f_sd = np.sqrt(R.std(axis=1, ddof=1) ** 2 + (np.sqrt(2) * ANALYTICAL_RSD * R.mean(axis=1)) ** 2)

bib = PROJECT_DIR / "results" / "bibliomic" / "bibliomicData_glutamatergicNeuron.xlsx"
p = culture_parameters(bib)
med = pd.read_excel(bib, sheet_name="mediaData")[["rxns", "mediumConcentrations"]].dropna().groupby("rxns")["mediumConcentrations"].max()
print(f"culture: V = {p['volume_L'] * 1e3:.1f} mL, dt = {p['interval_h']:.0f} h, gDW = {p['gDW'] * 1e3:.3f} mg  ->  "
      f"1 uM change = {p['k']:.4f} umol/gDW/h")

rows = []
for met, m in METABOLITE_MAP[METABOLITE_MAP.use].iterrows():
    if met not in D.index:
        continue
    c0 = med.get(m.rxns, np.nan)
    release = f[met] - f_sd[met] > 0
    absent = fresh[met] == 0 and (S.loc[met] > 0).all()
    r = dict(metabolite=met, rxns=m.rxns, fractionalChange=f[met], fractionalChangeSD=f_sd[met],
             mediumConc_uM=c0, note=m.note)
    if np.isfinite(f[met]) and np.isfinite(c0) and c0 > 0 and not release:
        r.update(mean=f[met] * c0 * p["k"], SD=f_sd[met] * c0 * p["k"])
        r["direction"] = "secreted" if r["mean"] - r["SD"] > 0 else "uptaken" if r["mean"] + r["SD"] < 0 else "unchanged"
    elif absent or release:
        r["direction"] = "secreted"
    elif np.isfinite(f[met]) and f[met] + f_sd[met] < 0:
        r["direction"] = "uptaken"
    else:
        r["direction"] = "unchanged"
    rows.append(r)
E = pd.DataFrame(rows)
print(E.direction.value_counts().to_string())

In [ ]:
recon = None
mapfile = DATA / "models" / "Recon3DModel_301_thermo.mat"
names = {}
if mapfile.exists():                      # metabolite IDs and names from Recon3D, for the output columns
    import scipy.io as sio
    m = sio.loadmat(mapfile, simplify_cells=True)
    M = m[[k for k in m if not k.startswith("__")][0]]
    names = dict(zip(M["mets"], M["metNames"]))
E["mets"] = E.rxns.str.replace(r"^EX_", "", regex=True)
E["metName"] = E.mets.map(names).fillna(E.metabolite)

quant = E[E["mean"].notna()].sort_values("rxns")
out = quant.assign(platform="HILIC-LC-MS")[["rxns", "mets", "metName", "platform", "mean", "SD"]]
out.to_csv(OMICS / "glutamatergic" / "exometabolomicData_glutamatergicNeuron.txt", index=False)
dirn = E[E["mean"].isna() & E.direction.isin(["secreted", "uptaken"])].sort_values(["direction", "rxns"])
dirn = dirn.assign(lb=np.where(dirn.direction == "secreted", 0.0, -10000.0),
                   ub=np.where(dirn.direction == "secreted", 10000.0, 0.0))
dirn[["rxns", "mets", "metName", "direction", "fractionalChange", "lb", "ub", "note"]].to_csv(
    OMICS / "glutamatergic" / "exometabolomicDirection_glutamatergicNeuron.txt", index=False)
E.to_csv(OMICS / "glutamatergic" / "raw" / "ST004403_processed.csv", index=False)
print(f"{len(out)} quantified exchange rates, {len(dirn)} direction-only exchanges")
out.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 0.28 * len(quant) + 1.2))
q = quant.sort_values("mean")
col = q.direction.map({"uptaken": "#2a78d6", "secreted": "#eb6834", "unchanged": "#9b9a96"})
ax.barh(q.metName, q["mean"], xerr=q["SD"], color=col, height=0.6, error_kw=dict(lw=0.8, color="#52514e"))
ax.axvline(0, color="#52514e", lw=0.8)
ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel("exchange flux (umol/gDW/h, symmetric log scale); negative = uptake")
ax.set_title("Glutamatergic neurons: quantified exchange rates (mean ± SD)")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout(); plt.show()

## 3. Caveats

* **Cholinergic culture.** The cholinergic cultures have a clear basal forebrain identity, but CHAT mRNA is low and there is a sizeable progenitor fraction. The curated `activeGenes`/`inactiveGenes` should keep priority over expression thresholds.
* **Glutamatergic rates.** The source study does not report medium volume, feeding interval or the composition of its commercial medium. The rates therefore assume that the same fraction of each nutrient is consumed in one 48-h interval in the lab medium.
* **Glucose.** Glucose is measured with low sensitivity in this assay, so its rate is uncertain.
* **Glutamine and alanine.** These are not used: the fresh medium contains an alanyl-glutamine dipeptide, and its hydrolysis raises both.
* **Cholinergic exometabolomics.** No exometabolomic data of human cholinergic neurons were found (Metabolomics Workbench, MetaboLights and PubMed, October 2026).